In [44]:
import concurrent.futures
import datetime as dt
import os
import sys
import time
import contextily as cnx
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import sliderule
import tqdm
from shapely.geometry import MultiPolygon, Polygon, box
from shapely.ops import orient
from tqdm import tqdm

import tiledb
from gedidb.providers.tiledb_provider import TileDBProvider

import gedidb as gdb

try:
    from dask.distributed import Client, LocalCluster
except ImportError:
    Client = None
    LocalCluster = None



In [36]:
#Read the Fire shapefile (boolean -> fire=1 , no fire=0). 
# It shows accumulated fire scars from 2019 - 2024. 
fire = gpd.read_file("Fire_2019_2024_vector.shp")

# Keep only burned areas (class 1)
fire_scars = fire[fire['Fire'] == 1] 

#Checking for the CRS (make sure it matches the one from fishnet)
fire.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [37]:
#Uploading the Arc of Deforestation outline
gf = gpd.read_file('AoD.geojson')

# Check CRS
gf.crs

#gf.explore()

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [38]:
# Get bounding box coordinates
minx, miny, maxx, maxy = gf.total_bounds

# Define grid size (adjust depending on CRS: degrees for EPSG:4326, meters for projected) 
cell_width = 0.5 # change this to control grid size 
cell_height = 0.5

# Generate rows and columns 
cols = np.arange(minx, maxx, cell_width) 
rows = np.arange(miny, maxy, cell_height) 

# Create polygons for each grid cell 
grid_cells = [] 
for x in cols: 
    for y in rows: 
        cell = box(x, y, x + cell_width, y + cell_height) 
        grid_cells.append(cell) 
        
# Convert to GeoDataFrame 
fishnet = gpd.GeoDataFrame({'geometry': grid_cells}, crs=gf.crs).drop_duplicates('geometry') 

fishnet = gpd.sjoin(fishnet, gf, how='inner').drop_duplicates('geometry') 

fishnet['tile_id']=fishnet.index

#ax = fishnet.boundary.plot() 
#sample = fishnet.iloc[[23]] 
#sample.geometry.plot(color='red', edgecolor='black', ax = ax, alpha=0.5) 
#gf.plot(ax=ax, color = 'yellow', alpha = 0.2) 

fishnet.explore()

In [39]:
#Using GEDIdb (GEDIProvider) to get Gedi data

provider = gdb.GEDIProvider(
    storage_type='s3',
    s3_bucket="dog.gedidb.gedi-l2-l4-v002",
    url="https://s3.gfz-potsdam.de"
)

In [40]:
# Define variables to query and quality filters
#https://gedidb.readthedocs.io/en/latest/user/tiledb_database.html

vars_selected = ["agbd", 'agbd_pi_lower', "agbd_pi_upper", "agbd_se", #biomass data
                 "l4_quality_flag", "predictor_limit_flag", #l4a filtering
                 "cover","rh_50","rh_90", "rh_98", "wsci", "pai", #structural metrics
                 "elev_lowestmode", "digital_elevation_model"] #terrain info

# Define quality filters
quality_filters = {
'sensitivity': '>= 0.98 and <= 1.0',
'beam_type': "== 'full'",
'l2a_quality_flag' : '== 1',
'degrade_flag': '== 0',
'surface_flag': '==1'
}

In [41]:
def patched_initialize_s3_context(self, credentials, url, region):
    config = {
        "vfs.s3.endpoint_override": url,
        "vfs.s3.region": region,
        "py.init_buffer_bytes": "2147483648",
        "sm.tile_cache_size": "2147483648",
        "sm.num_reader_threads": "4",
        "sm.num_tiledb_threads": "4",
        "vfs.s3.max_parallel_ops": "8",
        "vfs.s3.use_virtual_addressing": "true",
    }

    if credentials:
        config.update(
            {
                "vfs.s3.aws_access_key_id": credentials.get("AccessKeyId", ""),
                "vfs.s3.aws_secret_access_key": credentials.get("SecretAccessKey", ""),
                "vfs.s3.no_sign_request": "false",
            }
        )
    else:
        config["vfs.s3.no_sign_request"] = "true"

    return tiledb.Ctx(config)

TileDBProvider._initialize_s3_context = patched_initialize_s3_context

In [45]:
PROVIDER_KWARGS = {
    "storage_type": "s3",
    "s3_bucket": "dog.gedidb.gedi-l2-l4-v002",
    "url": "https://s3.gfz-potsdam.de",
}

start_time = "2019-01-01"
end_time = "2025-12-31"
parallel_backend = "threads"  
max_workers = 1
dask_memory_limit = "8GB"


def build_provider():
    return gdb.GEDIProvider(**PROVIDER_KWARGS)


import time

def process_tile(tile_row, max_retries=4, retry_wait=15):
    tile_id = tile_row.tile_id
    sq = gpd.GeoDataFrame({"geometry": [tile_row.geometry]}, crs=fishnet.crs).to_crs(4326)
    sq["geometry"] = sq["geometry"].simplify(0.001, preserve_topology=True)

    for attempt in range(1, max_retries + 1):
        try:
            provider = build_provider()

            rsps = provider.get_data(
                variables=vars_selected,
                query_type="bounding_box",
                geometry=sq,
                start_time=start_time,
                end_time=end_time,
                return_type="dataframe",
                **quality_filters,
            )

            if rsps is None or rsps.empty:
                return None
            
            # rsps["elevation_difference_tdx"] = rsps["elev_lowestmode"] - rsps["digital_elevation_model"]
            # rsps = rsps[rsps["elevation_difference_tdx"].between(-50, 50)].copy()

            if rsps.empty:
                return None

            rsps = gpd.GeoDataFrame(
                rsps,
                geometry=gpd.points_from_xy(rsps.longitude, rsps.latitude),
                crs="EPSG:4326",
            )

            rsps["time"] = pd.to_datetime(rsps["time"], errors="coerce")
            rsps["gedi_point_geom"] = rsps.geometry.copy()

            utm_crs = rsps.estimate_utm_crs()
            rsps_reprojected = rsps.to_crs(utm_crs)

            rsps_buffer_reprojected = rsps_reprojected.copy()
            rsps_buffer_reprojected["geometry"] = rsps_reprojected.buffer(22.7)

            rsps_shot = rsps_buffer_reprojected.to_crs(4326)

            fire_tile = fire_scars.clip(sq)
            if fire_tile.empty:
                return None

            rsps_fire = gpd.sjoin(
                rsps_shot,
                fire_tile[["geometry"]],
                how="inner",
                predicate="within",
            ).drop(columns=["index_right"], errors="ignore")

            if rsps_fire.empty:
                return None

            rsps_fire["tile_id"] = tile_id
            return rsps_fire

        except Exception as exc:
            print(f"Tile {tile_id} failed on attempt {attempt}/{max_retries}: {exc}")
            if attempt < max_retries:
                time.sleep(retry_wait)
            else:
                print(f"Skipping tile {tile_id} after {max_retries} failed attempts.")
                return None


def run_parallel_tiles(tile_rows, backend="threads", max_workers=4):
    backend = backend.lower()

    if backend == "serial":
        return [process_tile(tile_row) for tile_row in tqdm(tile_rows, desc="Processing tiles")]

    if backend == "threads":
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            return list(
                tqdm(
                    executor.map(process_tile, tile_rows),
                    total=len(tile_rows),
                    desc="Processing tiles (threads)",
                )
            )

    if backend == "processes":
        with concurrent.futures.ProcessPoolExecutor(max_workers=max_workers) as executor:
            return list(
                tqdm(
                    executor.map(process_tile, tile_rows),
                    total=len(tile_rows),
                    desc="Processing tiles (processes)",
                )
            )

    if backend == "dask":
        if Client is None or LocalCluster is None:
            raise ImportError("dask.distributed is not available in this environment.")

        cluster = LocalCluster(
            n_workers=max_workers,
            threads_per_worker=1,
            processes=True,
            memory_limit=DASK_MEMORY_LIMIT,
            dashboard_address=None,
        )
        client = Client(cluster)

        try:
            futures = client.map(process_tile, tile_rows)
            return list(tqdm(client.gather(futures), total=len(tile_rows), desc="Processing tiles (dask)"))
        finally:
            client.close()
            cluster.close()

    raise ValueError(f"Unsupported backend: {backend}")


# Use it to try different engines without changing the rest of the notebook.
tile_rows = [tile for tile in fishnet.itertuples(index=False)]
results = run_parallel_tiles(tile_rows, backend=parallel_backend, max_workers=max_workers)
valid_results = [result for result in results if result is not None and not result.empty]

if valid_results:
    all_rsps_fire = gpd.GeoDataFrame(pd.concat(valid_results, ignore_index=True), crs="EPSG:4326")
else:
    all_rsps_fire = gpd.GeoDataFrame(columns=["tile_id", "gedi_time", "geometry"], crs="EPSG:4326")

print(f"Completed {len(valid_results)} tiles with GEDI-fire matches using backend={parallel_backend!r}.")



Processing tiles (threads):  12%|█▏        | 41/342 [35:29<5:11:31, 62.10s/it]2026-03-19 15:50:31,772 - ERROR - Error querying TileDB array 's3://dog.gedidb.gedi-l2-l4-v002/array_uri': [TileDB::Array] Error: Caught std::exception: S3: Error while listing with prefix 's3://dog.gedidb.gedi-l2-l4-v002/array_uri/' and delimiter '/'[Error Type: 99] [HTTP Response Code: -1] [Remote IP: 139.17.228.42] : curlCode: 28, Timeout was reached; Details: Operation too slow. Less than 1 bytes/sec transferred the last 3 seconds


Tile 140 failed on attempt 1/4: [TileDB::Array] Error: Caught std::exception: S3: Error while listing with prefix 's3://dog.gedidb.gedi-l2-l4-v002/array_uri/' and delimiter '/'[Error Type: 99] [HTTP Response Code: -1] [Remote IP: 139.17.228.42] : curlCode: 28, Timeout was reached; Details: Operation too slow. Less than 1 bytes/sec transferred the last 3 seconds


Processing tiles (threads):  42%|████▏     | 145/342 [2:09:52<4:05:54, 74.90s/it]2026-03-19 17:25:14,184 - ERROR - Error querying TileDB array 's3://dog.gedidb.gedi-l2-l4-v002/array_uri': [TileDB::Task] Error: Caught std::exception: S3: Failed to read S3 object s3://dog.gedidb.gedi-l2-l4-v002/array_uri/__fragments/__1736887205464_1740318857372_28fcf8fdd819315e97919e6faf84072f_22/d1.tdb[Error Type: 99] [HTTP Response Code: 206] [Remote IP: 139.17.228.42] [Headers: 'accept-ranges' = 'bytes' 'content-length' = '8005020' 'content-range' = 'bytes 40025100-48030119/626789117' 'content-type' = 'application/octet-stream' 'date' = 'Fri, 20 Mar 2026 00:24:52 GMT' 'etag' = '"86c07b958bbd1f5ec038791f5158e4f3-12"' 'last-modified' = 'Mon, 24 Feb 2025 04:54:31 GMT' 'strict-transport-security' = 'max-age=63072000' 'x-amz-request-id' = 'tx0000089748161d9e4a337-0069bc93d4-e3d1746e-default' 'x-rgw-object-type' = 'Normal'] : curlCode: 28, Timeout was reached; Details: Operation too slow. Less than 1 bytes

Tile 353 failed on attempt 1/4: [TileDB::Task] Error: Caught std::exception: S3: Failed to read S3 object s3://dog.gedidb.gedi-l2-l4-v002/array_uri/__fragments/__1736887205464_1740318857372_28fcf8fdd819315e97919e6faf84072f_22/d1.tdb[Error Type: 99] [HTTP Response Code: 206] [Remote IP: 139.17.228.42] [Headers: 'accept-ranges' = 'bytes' 'content-length' = '8005020' 'content-range' = 'bytes 40025100-48030119/626789117' 'content-type' = 'application/octet-stream' 'date' = 'Fri, 20 Mar 2026 00:24:52 GMT' 'etag' = '"86c07b958bbd1f5ec038791f5158e4f3-12"' 'last-modified' = 'Mon, 24 Feb 2025 04:54:31 GMT' 'strict-transport-security' = 'max-age=63072000' 'x-amz-request-id' = 'tx0000089748161d9e4a337-0069bc93d4-e3d1746e-default' 'x-rgw-object-type' = 'Normal'] : curlCode: 28, Timeout was reached; Details: Operation too slow. Less than 1 bytes/sec transferred the last 3 seconds


Processing tiles (threads): 100%|██████████| 342/342 [4:33:22<00:00, 47.96s/it]  


Completed 332 tiles with GEDI-fire matches using backend='threads'.


In [46]:
all_rsps_fire_export = all_rsps_fire.copy()
all_rsps_fire_export["gedi_point_wkt"] = all_rsps_fire_export["gedi_point_geom"].to_wkt()
all_rsps_fire_export = all_rsps_fire_export.drop(columns=["gedi_point_geom"])

all_rsps_fire_export.to_file(
    "Gedi_fire_GEDIdb.gpkg",
    layer="Gedidb_fire",
    driver="GPKG"
)


2026-03-19 21:46:42,509 - INFO - Created 1,027,381 records


In [ ]:
all_rsps_fire_export.head()

,latitude,longitude,time,shot_number,digital_elevation_model,elev_lowestmode,cover,pai,l4_quality_flag,predictor_limit_flag,...,agbd_pi_upper,agbd_se,wsci,rh_50,rh_90,rh_98,elevation_difference_tdx,geometry,tile_id,gedi_point_wkt
0,-12.158787,-58.337588,2019-07-07,32021100100072101,334.519653,334.914276,0.003869,0.007753,1,255,...,14.567701,3.008365,6.630681,-0.18,1.42,2.17,0.394623,"POLYGON ((-58.33738 -12.15879, -58.33738 -12.1...",3,POINT (-58.337588 -12.158787)
1,-12.158373,-58.337276,2019-07-07,32021100100072102,335.252472,334.662109,0.003153,0.006316,1,255,...,14.567701,3.008365,6.562720,-0.18,1.38,2.17,-0.590363,"POLYGON ((-58.33707 -12.15837, -58.33707 -12.1...",3,POINT (-58.337276 -12.158373)
2,-12.157133,-58.336341,2019-07-07,32021100100072105,332.890411,331.544220,0.005046,0.010118,1,255,...,14.567701,3.008365,6.552817,-0.14,1.38,2.17,-1.346191,"POLYGON ((-58.33613 -12.15713, -58.33613 -12.1...",3,POINT (-58.336341 -12.157133)
3,-12.157546,-58.336652,2019-07-07,32021100100072104,332.890411,332.946167,0.005431,0.010891,1,255,...,14.420338,3.008637,6.605555,-0.22,1.34,2.13,0.055756,"POLYGON ((-58.33644 -12.15755, -58.33644 -12.1...",3,POINT (-58.336652 -12.157546)
4,-12.264433,-58.185237,2019-08-28,40191100400251104,312.646698,290.334534,0.913334,4.894983,0,255,...,123.747826,13.104303,9.326423,7.30,10.33,11.61,-22.312164,"POLYGON ((-58.18503 -12.26443, -58.18503 -12.2...",3,POINT (-58.185237 -12.264433)


In [ ]:
print("Rows, columns:", all_rsps_fire_export.shape)
print("\nColumn names:")
print(all_rsps_fire_export.columns.tolist())

print("\nData types:")
print(all_rsps_fire_export.dtypes)

print("\nGeoDataFrame info:")
print(all_rsps_fire_export.info())


Rows, columns: (1022794, 22)

Column names:
['latitude', 'longitude', 'time', 'shot_number', 'digital_elevation_model', 'elev_lowestmode', 'cover', 'pai', 'l4_quality_flag', 'predictor_limit_flag', 'agbd', 'agbd_pi_lower', 'agbd_pi_upper', 'agbd_se', 'wsci', 'rh_50', 'rh_90', 'rh_98', 'elevation_difference_tdx', 'geometry', 'tile_id', 'gedi_point_wkt']

Data types:
latitude                           float64
longitude                          float64
time                        datetime64[ns]
shot_number                         uint64
digital_elevation_model            float32
elev_lowestmode                    float32
cover                              float32
pai                                float32
l4_quality_flag                      uint8
predictor_limit_flag                 uint8
agbd                               float32
agbd_pi_lower                      float32
agbd_pi_upper                      float32
agbd_se                            float32
wsci                          